<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/bart_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BART baseline
Initial notebook setup.

# **Part 0.** Google Colab environment set up.

In [ ]:
# Reinstall ipython kernel to enable autoreload on latest runtime (by 02/12/2026)
# You'll be prompted to restart the session.
!pip install ipython==8.12.0

In [ ]:
!pip -q install transformers datasets sentencepiece accelerate

In [ ]:
import torch
import transformers
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import BartTokenizer, BartForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

torch.manual_seed(0)

In [ ]:
# Load Natural Questions (NQ) dataset
nq = load_dataset("sentence-transformers/natural-questions", split = "train[:20]")

print(nq)
print(nq.column_names)
print(nq[0])

# Now, we're going to add some helper functions to process data

In [ ]:
def get_question(data_line):
  return data_line['query']

def get_answer(data_line):
  answer = data_line['answer']
  if isinstance(answer, list):
    answer = answer[0]
  return answer

# Loading BART baseline

In [ ]:
# Load the BART baseline
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)

model = BartForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

print("Model loaded: ", model_name)

# Small test for a question from the data set

In [ ]:
question = "where does a lion live in the wild?"

inputs = tokenizer(
    question,
    return_tensors="pt",
    max_length=128,
    truncation=True
)
inputs = inputs.to(device)

with torch.no_grad():
  output = model.generate(
      **inputs,
      max_new_tokens = 16,
      num_beams = 4,
      no_repeat_ngram_size = 2,
  )

prediction = tokenizer.decode(output[0], skip_special_tokens=True)
print("Question: ", question)
print("Prediction: ", prediction)

# Test on the question from NQ dataset

In [ ]:
dataset_question = nq[0]
question = get_question(dataset_question)
answer = get_answer(dataset_question)


inputs = tokenizer(
    question,
    return_tensors="pt",
    max_length=128,
    truncation=True
)
inputs = inputs.to(device)

with torch.no_grad():
  output = model.generate(
      **inputs,
      max_new_tokens = 16,
      num_beams = 4,
      no_repeat_ngram_size = 2,
  )

prediction = tokenizer.decode(output[0], skip_special_tokens=True)
print("Question: ", question)
print("Answer: ", answer)
print("Prediction: ", prediction)


# Exact Match

In [ ]:
# Normalization
import re

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
# Check for Exact Match (EM)
answer_normalized = normalize_text(answer)
prediction_normalized = normalize_text(prediction)

em = int(answer_normalized == prediction_normalized)
print("EM: ", em)

# Test on 10 questions

In [ ]:
results = []

for i in range(10):
  dataset_question = nq[i]
  question = get_question(dataset_question)
  answer = get_answer(dataset_question)


  inputs = tokenizer(
      question,
      return_tensors="pt",
      max_length=128,
      truncation=True
  )
  inputs = inputs.to(device)

  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens = 16,
        num_beams = 4,
        no_repeat_ngram_size = 2,
    )

  prediction = tokenizer.decode(output[0], skip_special_tokens=True)

  answer_normalized = normalize_text(answer)
  prediction_normalized = normalize_text(prediction)

  em = (int)(answer_normalized==prediction_normalized)

  results.append({
      "question_number": i,
      "question": question,
      "answer": answer,
      "prediction": prediction,
      "em": em
  })

  df = pd.DataFrame(results)
  df.head()

# Calculating Correct Prediction Percentage

In [ ]:
accuracy = df['em'].sum()/len(df) * 100
print("Accuracy: ", accuracy)

# Saving Results

In [ ]:
df.to_csv('results.csv', index=False)
print("Results saved to results.csv")